## **MOVIE RECOMMENDER SYSTEM**

### **IMPORTING LIBRARIES**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

### **LOADING DATASET**

In [ ]:
movies = pd.read_csv("/content/movies.csv")
ratings = pd.read_csv("/content/ratings.csv")

### **DATA PREPROCESSING**

In [ ]:
print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [ ]:
print(ratings.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


*Checking for Null Values*

In [ ]:
movies.isnull().sum()

,0
movieId,0
title,0
genres,0


In [ ]:
ratings.isnull().sum()

,0
userId,0
movieId,0
rating,0
timestamp,0


In [ ]:
ratings.shape

(100836, 4)

In [ ]:
movies.shape

(9742, 3)

In [ ]:
ratings.dtypes

,0
userId,int64
movieId,int64
rating,float64
timestamp,int64


In [ ]:
movies.dtypes

,0
movieId,int64
title,object
genres,object


In [ ]:
movies.columns.tolist()

['movieId', 'title', 'genres']

In [ ]:
movies['genres'] = movies['genres'].fillna("")

### **TF-IDF VECTORIZATION**

In [ ]:
tfidf = TfidfVectorizer(stop_words= 'english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

In [ ]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23185 stored elements and shape (9742, 23)>
  Coords	Values
  (0, 1)	0.41684567364693936
  (0, 2)	0.5162254711770092
  (0, 3)	0.5048454681396087
  (0, 4)	0.26758647689140014
  (0, 8)	0.482990142708577
  (1, 1)	0.5123612074824269
  (1, 3)	0.6205251727456431
  (1, 8)	0.5936619434123594
  (2, 4)	0.5709154064399099
  (2, 18)	0.8210088907493954
  (3, 4)	0.5050154397005037
  (3, 18)	0.726240982959826
  (3, 7)	0.46640480307738325
  (4, 4)	1.0
  (5, 0)	0.5493281743985543
  (5, 5)	0.6359470441562757
  (5, 20)	0.5420423542868653
  (6, 4)	0.5709154064399099
  (6, 18)	0.8210088907493954
  (7, 1)	0.6366993258087036
  (7, 3)	0.7711121633813997
  (8, 0)	1.0
  (9, 1)	0.6295217016667962
  (9, 0)	0.5530653284926609
  (9, 20)	0.5457299419583338
  :	:
  (9731, 0)	0.41272965170024634
  (9731, 19)	0.508925697730817
  (9731, 9)	0.508925697730817
  (9732, 2)	0.5502833875552382
  (9732, 4)	0.28524046407869114
  (9732, 0)	0.39038039438445316
  (9732,

### **Cosine Simularity**

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

### **Recommendation Function**

In [ ]:
def recommend_movies(title, n=5):
  idx = indices[title] #Selects a movie
  sim_scores = list(enumerate(cosine_sim[idx])) #Set similarity score
  sim_scores = sorted(sim_scores, key= lambda x: x[1], reverse = True) #Sorting from High to Low
  sim_scores = sim_scores[1:n+1] #Select 5 similar movies
  movie_indices = [i[0] for i in sim_scores]
  return movies['title'].iloc[movie_indices].tolist()
print(recommend_movies("Grumpier Old Men (1995)", n=5))
print(recommend_movies("Jumanji (1995)", n=5))
print(recommend_movies("Heat (1995)", n=5))
print(recommend_movies("Toy Story (1995)", n=5))

['Sabrina (1995)', 'Clueless (1995)', 'Two if by Sea (1996)', 'French Twist (Gazon maudit) (1995)', 'If Lucy Fell (1996)']
['Indian in the Cupboard, The (1995)', 'NeverEnding Story III, The (1994)', 'Escape to Witch Mountain (1975)', "Darby O'Gill and the Little People (1959)", 'Return to Oz (1985)']
['Assassins (1995)', 'Die Hard: With a Vengeance (1995)', 'Net, The (1995)', 'Natural Born Killers (1994)', 'Judgment Night (1993)']
['Antz (1998)', 'Toy Story 2 (1999)', 'Adventures of Rocky and Bullwinkle, The (2000)', "Emperor's New Groove, The (2000)", 'Monsters, Inc. (2001)']


### **Ratings Dataset Preprocessing**

In [ ]:
ratings.columns.tolist()

['userId', 'movieId', 'rating', 'timestamp']

In [ ]:
ratings.isnull().sum()

,0
userId,0
movieId,0
rating,0
timestamp,0


In [ ]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
user_item_matrix = ratings.pivot_table(index = 'userId', columns= 'movieId', values= 'rating')
user_item_matrix.fillna(0, inplace=True)

In [ ]:
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity,
                           index = user_item_matrix.columns,
                           columns= user_item_matrix.columns)

In [ ]:
def recommend_movies_item_based(movie_id, n=5):
  sim_scores = item_similarity_df[movie_id].sort_values(ascending=False)
  sim_scores = sim_scores.iloc[1:n+1]
  recommended_titles = movies[movies['movieId'].isin(sim_scores.index)]['title'].tolist()
  return recommended_titles
print(recommend_movies_item_based(1))

['Star Wars: Episode IV - A New Hope (1977)', 'Forrest Gump (1994)', 'Jurassic Park (1993)', 'Independence Day (a.k.a. ID4) (1996)', 'Toy Story 2 (1999)']


### **Collaborative Filtering**

In [ ]:
cf_user_item_matrix = ratings.pivot(index ='userId', columns='movieId', values='rating')
cf_user_item_matrix.fillna(0, inplace = True)

In [ ]:
cf_user_sim = cosine_similarity(cf_user_item_matrix)
cf_user_sim_df = pd.DataFrame(
    cf_user_sim,
    index = cf_user_item_matrix.index,
    columns = user_item_matrix.index
)

In [ ]:
def recommend_movies_based_on_user(user_id, n=5):
  sim_users = cf_user_sim_df[user_id].sort_values(ascending= False)
  sim_users = sim_users.iloc[1:n+1]
  similar_users = sim_users.index
  sim_user_ratings = cf_user_item_matrix.loc[similar_users].mean(axis=0)
  target_user_rated = cf_user_item_matrix.loc[user_id]
  recommendations = sim_user_ratings[target_user_rated == 0].sort_values(ascending = False)

  recommended_title = movies[movies['movieId'].isin(recommendations.index)]['title'].tolist()
  return recommended_title[:n]
print(recommend_movies_based_on_user(3, n=4))

['Toy Story (1995)', 'Jumanji (1995)', 'Grumpier Old Men (1995)', 'Waiting to Exhale (1995)']


### **Evaluation Metrics**

In [ ]:
movie_mean = ratings.groupby('movieId')['rating'].mean()
y_true = ratings['rating']
y_pred = ratings['movieId'].map(movie_mean)

In [ ]:
rmse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
print("RMSE : ", rmse)
print("MAE : ", mae)

RMSE :  0.767678278417834
MAE :  0.6667569987497155


### **Precision@k and Recall@k**

In [ ]:
def precision_at_k(recommended, relevant, k):
  recommended_at_k_precision = recommended[:k]#top k recommendations
  relevant_rec = set(recommended_at_k_precision) & set(relevant) #converts into sets and check for intersection
  return len(relevant_rec) / k if k != 0 else 0

def recall_at_k(recommended, relevant, k):
  recommended_at_k_recall = recommended[:k]
  relevant_recs = set(recommended_at_k_recall) & set(relevant)
  return len(relevant_recs) / len(relevant) if len(relevant) != 0 else 0

In [ ]:
user_id = 1

In [ ]:
relevant = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4.0)]['movieId'].tolist()
recommended = [1, 50, 32, 260, 480]
k = 5
print("Precision@5 : ", precision_at_k(recommended, relevant, k))
print("Recall@5 : ", recall_at_k(recommended, relevant, k))

Precision@5 :  0.8
Recall@5 :  0.02


### **Average of Precision and Recall**

In [ ]:
all_precisions = []
all_recalls = []
for uid in ratings['userId'].unique():
  relevant = ratings[(ratings['userId'] == uid) & (ratings['rating'] >= 4.0)]['movieId'].tolist()
  recommended = [1, 50, 32, 260, 480]

  if relevant :
    all_precisions.append(precision_at_k(recommended, relevant, 5))
    all_recalls.append(recall_at_k(recommended, relevant,5))
print("Avg Precision@9: ", np.mean(all_precisions))
print("Avg Recall@9: ", np.mean(all_recalls))

Avg Precision@9:  0.2541871921182266
Avg Recall@9:  0.026102084781575254


### **Hybrid Recommendations**

In [ ]:
def hybrid_recommendation(user_id, movie_title, alpha = 0.5, n=5):
  idx = indices[movie_title]
  content_scores = pd.Series(cosine_sim[idx], index=movies['movieId'])
  cf_scores = item_similarity_df.loc[movies.loc[idx, 'movieId']]
  combined = pd.concat([content_scores, cf_scores], axis=1, join='inner')
  combined.columns = ['content', 'cf']
  final_scores = alpha * combined['content'] + (1 - alpha) * combined['cf']
  top_movies = final_scores.sort_values(ascending=False).iloc[1:n+1].index
  return movies[movies['movieId'].isin(top_movies)]['title'].tolist()

user_id = 1
movie_title = 'Toy Story (1995)'

hyb_recommendations = hybrid_recommendation(user_id, movie_title, alpha = 0.5, n=5)
print(f"Input Movie : {movie_title}")
print("Top 5 Hybrid Recommendations: ")
for i, rec in enumerate(hyb_recommendations, 1):
  print(f" {i}.{rec}")

Input Movie : Toy Story (1995)
Top 5 Hybrid Recommendations: 
 1.Antz (1998)
 2.Toy Story 2 (1999)
 3.Shrek (2001)
 4.Monsters, Inc. (2001)
 5.Finding Nemo (2003)
